In [1]:
pip install groq

#

In [2]:
# Import the Groq library and securely load the API key
from groq import Groq
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# Initialize the Groq client
client = Groq(
    api_key=GROQ_API_KEY,
)

Now you can use the `client` object to interact with the Groq API. Here's a simple example of generating text:

In [3]:
# This cell was for testing the Groq client and is no longer needed in the main flow.

In [4]:
pip install pypdf sentence-transformers

Now that the libraries are installed, let's load the PDF document and extract its text content. We will then break the text into smaller, overlapping chunks. This is crucial for Retrieval Augmented Generation (RAG) as it allows the model to retrieve specific, relevant information without processing the entire document at once, and also helps in fitting the context window of the LLM.

In [5]:
from pypdf import PdfReader

def get_pdf_text(pdf_path):
    text = ""
    pdf_reader = PdfReader(pdf_path)
    for page in pdf_reader.pages:
        text += page.extract_text()
    return text

# Load the Nestlé HR policy PDF
pdf_path = '/content/the_nestle_hr_policy_pdf_2012.pdf'
raw_text = get_pdf_text(pdf_path)
print(f"Extracted {len(raw_text)} characters from the PDF.")
# print(raw_text[:500]) # Display first 500 characters to verify

Extracted 14318 characters from the PDF.


In [11]:
pip install langchain


In [13]:
pip install langchain-text-splitters

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def get_text_chunks(text):
    # Using RecursiveCharacterTextSplitter for robust text splitting
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    chunks = text_splitter.split_text(text)
    return chunks

# Get text chunks
text_chunks = get_text_chunks(raw_text)
print(f"Created {len(text_chunks)} text chunks.")
# print(text_chunks[0]) # Display the first chunk to verify

Created 18 text chunks.


Next, we'll convert these text chunks into numerical vectors (embeddings). These embeddings will allow us to find the most relevant document sections when a user asks a question. We'll use a `SentenceTransformer` model for this purpose.

In [25]:
from sentence_transformers import SentenceTransformer
import numpy as np

def get_embeddings(text_chunks):
    # Initialize an embedding model. A good open-source choice is 'all-MiniLM-L6-v2'
    # which is efficient and performs well for many tasks.
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = embedding_model.encode(text_chunks)
    return embeddings

# Generate embeddings for each text chunk
document_embeddings = get_embeddings(text_chunks)
print(f"Generated embeddings for {len(document_embeddings)} chunks.")
print(f"Shape of one embedding: {document_embeddings[0].shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generated embeddings for 18 chunks.
Shape of one embedding: (384,)


In [26]:
pip install chromadb

With `chromadb` installed, we can now create a persistent vector store. This will store our text chunks and their corresponding embeddings, allowing us to quickly search for relevant information based on a user's query.

In [27]:
import chromadb
from chromadb.utils import embedding_functions

# Initialize ChromaDB client (in-memory for this example, or persistent if you prefer)
chroma_client = chromadb.Client()

# Or for a persistent client:
# chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Use the same SentenceTransformer model for query embeddings
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# Create a collection
collection_name = "nestle_hr_policy"

try:
    collection = chroma_client.get_collection(name=collection_name)
    print(f"Collection '{collection_name}' already exists. Using existing collection.")
except:
    collection = chroma_client.create_collection(
        name=collection_name,
        embedding_function=embedding_function # Pass the embedding function directly
    )
    # Prepare documents for ChromaDB
    documents = text_chunks
    metadatas = [{"(chunk_id)": str(i)} for i in range(len(text_chunks))]
    ids = [f"chunk_{i}" for i in range(len(text_chunks))]

    # Add the documents to the collection
    collection.add(
        documents=documents,
        metadatas=metadatas,
        ids=ids
    )
    print(f"Created new collection '{collection_name}' and added {len(text_chunks)} documents.")

print(f"Number of items in collection: {collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Created new collection 'nestle_hr_policy' and added 18 documents.
Number of items in collection: 18


Now that our vector store is set up, we need a way to retrieve the most relevant chunks given a user's query. We'll define a function that takes a query, generates its embedding, and then queries the ChromaDB collection to find the top `k` most similar document chunks.

In [28]:
def retrieve_info(query, collection, top_k=5):
    # Query the collection for similar documents
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )
    # Extract the documents (text chunks) from the results
    retrieved_documents = results['documents'][0]
    return retrieved_documents

# Test the retrieval function
# test_query = "What is the policy on annual leave?"
# relevant_chunks = retrieve_info(test_query, collection)
# print(f"Relevant chunks for query '{test_query}':")
# for i, chunk in enumerate(relevant_chunks):
#     print(f"--- Chunk {i+1} ---")
#     print(chunk[:200] + '...') # Print first 200 characters of each chunk

Now, let's create a prompt template that will guide the Groq model in generating responses. This template will incorporate the retrieved context and the user's question, ensuring the model provides relevant and accurate answers.

In [29]:
def get_groq_chat_completion(client, query: str, context: list):
    # Combine the retrieved context into a single string
    context_str = "\n\n".join(context)

    # Define the prompt template for the Groq model
    prompt = f"""You are a helpful AI assistant that answers questions about the provided HR policy document. \n\nUse the following pieces of context to answer the user's question. \nIf you don't know the answer, just say that you don't know, don't try to make up an answer.\n--------------------\nCONTEXT: {context_str}\n--------------------\nQUESTION: {query}\nANSWER:"""

    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are a helpful AI assistant specialized in Nestlé HR policies. Provide concise and accurate answers based only on the provided context.",
                },
                {
                    "role": "user",
                    "content": prompt,
                },
            ],
            model="llama3-8b-8192", # Using a generally available 8B model
            temperature=0.7,
            max_tokens=500,
            top_p=1,
            stop=None,
            stream=False,
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        print(f"Error generating chat completion: {e}")
        return "I apologize, but I encountered an error while trying to generate a response. Please try again."

# # Test the response generation function
# test_query = "What is the policy on annual leave?"
# retrieved_chunks = retrieve_info(test_query, collection)
# response = get_groq_chat_completion(client, test_query, retrieved_chunks)
# print(f"\nQuestion: {test_query}")
# print(f"Answer: {response}")

Finally, let's integrate all these components into a user-friendly chatbot interface using Gradio. This will allow users to ask questions and receive answers directly from our RAG system.

In [30]:
pip install gradio

In [31]:
import gradio as gr

def chatbot_interface(user_query):
    # 1. Retrieve relevant information from the vector store
    retrieved_chunks = retrieve_info(user_query, collection, top_k=5)

    # 2. Generate a response using the Groq API and the retrieved chunks
    response = get_groq_chat_completion(client, user_query, retrieved_chunks)

    return response

# Create the Gradio interface
iface = gr.Interface(
    fn=chatbot_interface,
    inputs=gr.Textbox(lines=2, placeholder="Ask a question about the Nestlé HR policy..."),
    outputs="text",
    title="Nestlé HR Policy Chatbot (Powered by Groq)",
    description="Ask me anything about Nestlé's HR policy document!"
)

# Launch the Gradio interface
iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c8a7a80c5798a15a2a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
